## Download and Index WQP

In [ ]:
import os,sys,psycopg2,tempfile,git;
import requests,csv,codecs,datetime;
from contextlib import closing;

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

dbse = os.environ['POSTGRESQL_DB'];
host = os.environ['POSTGRESQL_HOST'];
port = os.environ['POSTGRESQL_PORT'];
user = 'cipsrv';
pswd = os.environ['POSTGRESQL_CIP_PASS'];

extract_date = '20260402';
loader_id    = 'PDZIEMIE';

print("extract date " + str(extract_date) + " by " + str(loader_id));
print(f"Start: {datetime.datetime.now().strftime('%H:%M:%S')}");

In [ ]:
%%time

cs = "dbname=%s user=%s password=%s host=%s port=%s" % (
     dbse
    ,user
    ,pswd
    ,host
    ,port
);

try:
    conn = psycopg2.connect(cs);
except:
    raise Exception("database connection error");

print("database is ready");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_stations_" + extract_date);
    
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_stations_""" + extract_date + """(
    organizationidentifier                          VARCHAR
   ,organizationformalname                          VARCHAR
   ,monitoringlocationidentifier                    VARCHAR
   ,monitoringlocationname                          VARCHAR
   ,monitoringlocationtypename                      VARCHAR
   ,monitoringlocationdescriptiontext               VARCHAR
   ,huceightdigitcode                               VARCHAR
   ,drainageareameasure_measurevalue                VARCHAR
   ,drainageareameasure_measureunitcode             VARCHAR
   ,contributingdrainageareameasure_measurevalue    VARCHAR
   ,contributingdrainageareameasure_measureunitcode VARCHAR
   ,latitudemeasure                                 VARCHAR
   ,longitudemeasure                                VARCHAR
   ,sourcemapscalenumeric                           VARCHAR
   ,horizontalaccuracymeasure_measurevalue          VARCHAR
   ,horizontalaccuracymeasure_measureunitcode       VARCHAR
   ,horizontalcollectionmethodname                  VARCHAR
   ,horizontalcoordinatereferencesystemdatumname    VARCHAR
   ,verticalmeasure_measurevalue                    VARCHAR
   ,verticalmeasure_measureunitcode                 VARCHAR
   ,verticalaccuracymeasure_measurevalue            VARCHAR
   ,verticalaccuracymeasure_measureunitcode         VARCHAR
   ,verticalcollectionmethodname                    VARCHAR
   ,verticalcoordinatereferencesystemdatumname      VARCHAR
   ,countrycode                                     VARCHAR
   ,statecode                                       VARCHAR
   ,countycode                                      VARCHAR
   ,aquifername                                     VARCHAR
   ,localaqfrname                                   VARCHAR
   ,formationtypetext                               VARCHAR
   ,aquifertypename                                 VARCHAR
   ,constructiondatetext                            VARCHAR
   ,welldepthmeasure_measurevalue                   VARCHAR
   ,welldepthmeasure_measureunitcode                VARCHAR
   ,wellholedepthmeasure_measurevalue               VARCHAR
   ,wellholedepthmeasure_measureunitcode            VARCHAR
   ,providername                                    VARCHAR
);
    """);
    
    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

target = os.path.join('/tmp','wqp_harvest_'+ extract_date + '.csv');

if os.path.exists(target):
    os.remove(target);

r = requests.get('https://www.waterqualitydata.us/Codes/countrycode?mimeType=json');
r_json = r.json();
countries = [];
for item in r_json["codes"]:
    if item["value"] != "US":
        countries.append(item["value"]);
 
r = requests.get('http://www.waterqualitydata.us/Codes/statecode?countrycode=US&mimeType=json');
r_json = r.json();
states = [];
for item in r_json["codes"]:
    states.append(item["value"]);
   
isFirst = True;
totcnt = 0;
with open(target,'w',newline='',encoding='utf-8') as f:
    writer = csv.writer(f,delimiter=',',strict=True);
   
    for state in states:
        print(state,end="");
        cnt = 0;
      
        payload = {'statecode':state,'mimeType':'csv','sorted':'no'};
        r = requests.get('https://www.waterqualitydata.us/data/Station/search',params=payload,stream=True);
        rd = [line.decode('utf-8') for line in r.iter_lines()];
        cr = csv.reader(rd);
      
        header = True;
        for row in cr:
      
            if header:
                if isFirst:
                    for idx,ch in enumerate(row):
                        row[idx] = row[idx].replace('/','_');
               
                    writer.writerow(row);
            
            else:
                for idx,ch in enumerate(row):
                    row[idx] = row[idx].replace(u'\u00b6','');
               
                    if row[idx].find("'") > 0:
                        row[idx] = row[idx].replace("'","''");
            
                writer.writerow(row);
                cnt = cnt + 1;
            
            header = False;
            
        isFirst = False;
        print(" - " + str(cnt));
        totcnt = totcnt + cnt;
      
    for country in countries:
        print(country,end="");
        cnt = 0;
      
        payload = {'countrycode':country,'mimeType':'csv','sorted':'no'};
        r = requests.get('https://www.waterqualitydata.us/data/Station/search',params=payload,stream=True);
        rd = [line.decode('utf-8') for line in r.iter_lines()];
        cr = csv.reader(rd);
      
        header = True;
        for row in cr:
      
            if header and isFirst:
                for idx,ch in enumerate(row):
                    row[idx] = row[idx].replace('/','_');
               
                writer.writerow(row);
            
            else:
                for idx,ch in enumerate(row):
                    row[idx] = row[idx].replace(u'\u00b6','');
               
                writer.writerow(row);
                cnt = cnt + 1;
            
            header = False;
            
        isFirst = False;
        print(" - " + str(cnt));
        totcnt = totcnt + cnt;

print("downloaded " + str(totcnt) + " records from WQP.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("TRUNCATE TABLE cipdev_owld.wqp_stations_" + extract_date);
    conn.commit();
    
    with open(target,'r') as f:
        cursor.copy_expert("COPY cipdev_owld.wqp_stations_" + extract_date + " FROM STDIN WITH (FORMAT CSV, HEADER)",f);   
    conn.commit();

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_stations_" + extract_date);
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records to CIP-service database.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_stations_seq"); 
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_stations_seq START WITH 1");

    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_stations");
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_stations(
    objectid                                        INTEGER
   ,organizationidentifier                          VARCHAR(256)
   ,organizationformalname                          VARCHAR(256)
   ,monitoringlocationidentifier                    VARCHAR(256)
   ,monitoringlocationname                          VARCHAR(768)
   ,monitoringlocationtypename                      VARCHAR(256)
   ,monitoringlocationdescriptiontext               VARCHAR(4000)
   ,huceightdigitcode                               VARCHAR(8)
   ,drainageareameasure_measurevalue                NUMERIC
   ,drainageareameasure_measureunitcode             VARCHAR(16)
   ,contributingdrainageareameasure_measurevalue    NUMERIC
   ,contributingdrainageareameasure_measureunitcode VARCHAR(16)
   ,latitudemeasure                                 NUMERIC
   ,longitudemeasure                                NUMERIC
   ,sourcemapscalenumeric                           VARCHAR(64)
   ,horizontalaccuracymeasure_measurevalue          VARCHAR(64)
   ,horizontalaccuracymeasure_measureunitcode       VARCHAR(16)
   ,horizontalcollectionmethodname                  VARCHAR(2000)
   ,horizontalcoordinatereferencesystemdatumname    VARCHAR(16)
   ,verticalmeasure_measurevalue                    VARCHAR(64)
   ,verticalmeasure_measureunitcode                 VARCHAR(16)
   ,verticalaccuracymeasure_measurevalue            VARCHAR(64)
   ,verticalaccuracymeasure_measureunitcode         VARCHAR(16)
   ,verticalcollectionmethodname                    VARCHAR(2000)
   ,verticalcoordinatereferencesystemdatumname      VARCHAR(16)
   ,countrycode                                     VARCHAR(2)
   ,statecode                                       VARCHAR(2)
   ,countycode                                      VARCHAR(3)
   ,aquifername                                     VARCHAR(2000)
   ,localaqfrname                                   VARCHAR(2000)
   ,formationtypetext                               VARCHAR(2000)
   ,aquifertypename                                 VARCHAR(2000)
   ,constructiondatetext                            VARCHAR(16)
   ,welldepthmeasure_measurevalue                   NUMERIC
   ,welldepthmeasure_measureunitcode                VARCHAR(16)
   ,wellholedepthmeasure_measurevalue               NUMERIC
   ,wellholedepthmeasure_measureunitcode            VARCHAR(16)
   ,providername                                    VARCHAR(16)
   ,shape                                           GEOMETRY
)
    """);
    cursor.execute("ALTER TABLE IF EXISTS cipdev_owld.wqp_stations ADD CONSTRAINT wqp_stations_pk PRIMARY KEY (monitoringlocationidentifier)");
    cursor.execute("ALTER TABLE IF EXISTS cipdev_owld.wqp_stations ADD CONSTRAINT wqp_stations_u01 UNIQUE (objectid)");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("""
INSERT INTO cipdev_owld.wqp_stations(
    objectid
   ,organizationidentifier
   ,organizationformalname
   ,monitoringlocationidentifier
   ,monitoringlocationname
   ,monitoringlocationtypename
   ,monitoringlocationdescriptiontext
   ,huceightdigitcode
   ,drainageareameasure_measurevalue
   ,drainageareameasure_measureunitcode
   ,contributingdrainageareameasure_measurevalue
   ,contributingdrainageareameasure_measureunitcode
   ,latitudemeasure
   ,longitudemeasure
   ,sourcemapscalenumeric
   ,horizontalaccuracymeasure_measurevalue
   ,horizontalaccuracymeasure_measureunitcode
   ,horizontalcollectionmethodname
   ,horizontalcoordinatereferencesystemdatumname
   ,verticalmeasure_measurevalue
   ,verticalmeasure_measureunitcode
   ,verticalaccuracymeasure_measurevalue
   ,verticalaccuracymeasure_measureunitcode
   ,verticalcollectionmethodname
   ,verticalcoordinatereferencesystemdatumname
   ,countrycode
   ,statecode
   ,countycode
   ,aquifername
   ,localaqfrname
   ,formationtypetext
   ,aquifertypename
   ,constructiondatetext
   ,welldepthmeasure_measurevalue
   ,welldepthmeasure_measureunitcode
   ,wellholedepthmeasure_measurevalue
   ,wellholedepthmeasure_measureunitcode
   ,providername
   ,shape
)
SELECT
 NEXTVAL('cipdev_owld.wqp_stations_seq')::INTEGER AS objectid
,a.organizationidentifier
,a.organizationformalname
,a.monitoringlocationidentifier
,a.monitoringlocationname
,a.monitoringlocationtypename
,a.monitoringlocationdescriptiontext
,a.huceightdigitcode
,a.drainageareameasure_measurevalue
,a.drainageareameasure_measureunitcode
,a.contributingdrainageareameasure_measurevalue
,a.contributingdrainageareameasure_measureunitcode
,a.latitudemeasure
,a.longitudemeasure
,a.sourcemapscalenumeric
,a.horizontalaccuracymeasure_measurevalue
,a.horizontalaccuracymeasure_measureunitcode
,a.horizontalcollectionmethodname
,a.horizontalcoordinatereferencesystemdatumname
,a.verticalmeasure_measurevalue
,a.verticalmeasure_measureunitcode
,a.verticalaccuracymeasure_measurevalue
,a.verticalaccuracymeasure_measureunitcode
,a.verticalcollectionmethodname
,a.verticalcoordinatereferencesystemdatumname
,a.countrycode
,a.statecode
,a.countycode
,a.aquifername
,a.localaqfrname
,a.formationtypetext
,a.aquifertypename
,a.constructiondatetext
,a.welldepthmeasure_measurevalue
,a.welldepthmeasure_measureunitcode
,a.wellholedepthmeasure_measurevalue
,a.wellholedepthmeasure_measureunitcode
,a.providername
,CASE
 WHEN a.latitudemeasure  IS NULL
 OR   a.longitudemeasure IS NULL
 OR   a.latitudemeasure = 0
 OR   longitudemeasure  = 0
 THEN
   CAST(NULL AS GEOMETRY)
 ELSE
   CASE
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'WGS84'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4326)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'WGS72'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4322)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'OLDHI'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4135)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'GUAM'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4675)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'WAKE'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4733)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'AMSMA'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4169)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'PR'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4139)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'ASTRO'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4715)
         ,4269
      )
   
   WHEN a.horizontalcoordinatereferencesystemdatumname = 'NAD27'
   THEN
      public.ST_TRANSFORM(
          public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4267)
         ,4269
      )
   
   ELSE
      public.ST_POINT(a.longitudemeasure,a.latitudemeasure,4269)

   END
 END AS shape
FROM (
   SELECT
    aa.organizationidentifier
   ,aa.organizationformalname
   ,aa.monitoringlocationidentifier
   ,aa.monitoringlocationname
   ,aa.monitoringlocationtypename
   ,aa.monitoringlocationdescriptiontext
   ,aa.huceightdigitcode
   ,aa.drainageareameasure_measurevalue::NUMERIC AS drainageareameasure_measurevalue 
   ,aa.drainageareameasure_measureunitcode
   ,aa.contributingdrainageareameasure_measurevalue::NUMERIC AS contributingdrainageareameasure_measurevalue
   ,aa.contributingdrainageareameasure_measureunitcode
   ,aa.latitudemeasure::NUMERIC  AS latitudemeasure 
   ,aa.longitudemeasure::NUMERIC AS longitudemeasure
   ,REPLACE(
       REGEXP_REPLACE(REPLACE(REPLACE(aa.sourcemapscalenumeric,'GPS-Unspecified',NULL),'NA',NULL),'^1\\:','')
      ,','
      ,''
    )::NUMERIC AS sourcemapscalenumeric
   ,aa.horizontalaccuracymeasure_measurevalue
   ,aa.horizontalaccuracymeasure_measureunitcode
   ,aa.horizontalcollectionmethodname
   ,aa.horizontalcoordinatereferencesystemdatumname
   ,aa.verticalmeasure_measurevalue
   ,aa.verticalmeasure_measureunitcode
   ,aa.verticalaccuracymeasure_measurevalue
   ,aa.verticalaccuracymeasure_measureunitcode
   ,aa.verticalcollectionmethodname
   ,aa.verticalcoordinatereferencesystemdatumname
   ,aa.countrycode
   ,aa.statecode
   ,aa.countycode
   ,aa.aquifername
   ,aa.localaqfrname
   ,aa.formationtypetext
   ,aa.aquifertypename
   ,aa.constructiondatetext
   ,aa.welldepthmeasure_measurevalue::NUMERIC
   ,aa.welldepthmeasure_measureunitcode
   ,aa.wellholedepthmeasure_measurevalue::NUMERIC
   ,aa.wellholedepthmeasure_measureunitcode
   ,aa.providername
   FROM
   cipdev_owld.wqp_stations_20260402 aa
   WHERE
   aa.organizationidentifier != 'OrganizationIdentifier'
) a
    """);
    conn.commit();
    cursor.execute("ANALYZE cipdev_owld.wqp_stations");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_stations");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_stations.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_control");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_control(
    objectid        INTEGER     NOT NULL
   ,keyword         VARCHAR     NOT NULL
   ,value_str       VARCHAR
   ,value_num       NUMERIC
   ,value_date      DATE
   ,globalid        VARCHAR(40) NOT NULL
   ,CONSTRAINT wqp_control_u01 UNIQUE (objectid)
   ,CONSTRAINT wqp_control_u02 UNIQUE (globalid)
)
    """);

    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (1,'NAME'       ,'Water Quality Portal Stations',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (2,'DESCRIPTION','Water Quality Portal stations indexed to medium and high resolution NHDPlus catchments and reach codes.',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (3,'URL'        ,'https://www.waterqualitydata.us/',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (4,'EVENTTYPE'  ,NULL,10032,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (5,'VINTAGE'    ,NULL,NULL,TO_DATE('" + extract_date + "','YYYYMMDD'),'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (6,'RESOLUTION' ,'HR',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (7,'RESOLUTION' ,'MR',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (8,'PRECISION'  ,'catchment',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (9,'PRECISION'  ,'reach measures',NULL,NULL,'{' || uuid_generate_v1() || '}')");
    cursor.execute("INSERT INTO cipdev_owld.wqp_control(objectid, keyword, value_str, value_num, value_date, globalid) VALUES (10,'XWALK_HUC12_VERSION','NP21',NULL,NULL,'{' || uuid_generate_v1() || '}')");

    conn.commit();

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_control");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_control.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_sfid");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_sfid(
    objectid               INTEGER NOT NULL
   ,source_originator      VARCHAR(130) NOT NULL
   ,source_featureid       VARCHAR(100) NOT NULL
   ,source_featureid2      VARCHAR(100)
   ,source_series          VARCHAR(100)
   ,source_subdivision     VARCHAR(100)
   ,source_joinkey         VARCHAR(40) NOT NULL
   ,start_date             DATE
   ,end_date               DATE
   ,sfiddetailurl          VARCHAR(255)
   ,load_id                VARCHAR(255)
   ,load_date              DATE
   ,src_event_count        INTEGER
   ,src_point_count        INTEGER
   ,src_line_count         INTEGER
   ,src_area_count         INTEGER
   ,cat_mr_count           INTEGER
   ,cat_hr_count           INTEGER
   ,xwalk_huc12_np21_count INTEGER
   ,xwalk_huc12_nphr_count INTEGER
   ,xwalk_huc12_f3_count   INTEGER
   ,rad_mr_event_count     INTEGER
   ,rad_hr_event_count     INTEGER
   ,rad_mr_point_count     INTEGER
   ,rad_hr_point_count     INTEGER
   ,rad_mr_line_count      INTEGER
   ,rad_hr_line_count      INTEGER
   ,rad_mr_area_count      INTEGER
   ,rad_hr_area_count      INTEGER
   ,globalid               VARCHAR(40) NOT NULL
   ,shape                  GEOMETRY
   ,CONSTRAINT wqp_sfid_pk PRIMARY KEY (source_joinkey)
)
    """);

    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i01 ON cipdev_owld.wqp_sfid(source_originator)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i02 ON cipdev_owld.wqp_sfid(source_featureid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i03 ON cipdev_owld.wqp_sfid(source_featureid2)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i04 ON cipdev_owld.wqp_sfid(src_event_count)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i05 ON cipdev_owld.wqp_sfid(rad_mr_event_count)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_i06 ON cipdev_owld.wqp_sfid(rad_hr_event_count)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_sfid_pk2 ON cipdev_owld.wqp_sfid(source_originator,source_featureid,source_featureid2,source_series,start_date)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_sfid_spx ON cipdev_owld.wqp_sfid USING gist(shape)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_sfid_u01 ON cipdev_owld.wqp_sfid(objectid)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_sfid_u02 ON cipdev_owld.wqp_sfid(globalid)");
    
    cursor.execute("""
INSERT INTO cipdev_owld.wqp_sfid(
    objectid
   ,source_originator
   ,source_featureid
   ,source_featureid2
   ,source_joinkey
   ,start_date
   ,load_id
   ,load_date
   ,src_event_count
   ,src_point_count
   ,src_line_count
   ,src_area_count
   ,globalid
)
SELECT
 a.objectid
,a.organizationidentifier
,a.monitoringlocationidentifier
,a.providername
,'{' || uuid_generate_v1() || '}'
,TO_DATE('""" + extract_date + """','YYYYMMDD')
,'""" + loader_id + """'
,TO_DATE('""" + extract_date + """','YYYYMMDD')
,CASE WHEN a.shape IS NULL
 THEN
   0
 ELSE
   1
 END AS src_event_count
,CASE WHEN a.shape IS NULL
 THEN
   0
 ELSE
   1
 END AS src_point_count
,0 AS src_line_count
,0 AS src_area_count
,'{' || uuid_generate_v1() || '}'
FROM
cipdev_owld.wqp_stations a
    """);

    conn.commit();
    cursor.execute("ANALYZE cipdev_owld.wqp_sfid");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_sfid");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_sfid.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_src_p");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_src_p
(
    objectid           INTEGER NOT NULL
   ,permid_joinkey     VARCHAR(40) NOT NULL
   ,source_originator  VARCHAR(130) NOT NULL
   ,source_featureid   VARCHAR(100) NOT NULL
   ,source_featureid2  VARCHAR(100)
   ,source_series      VARCHAR(100)
   ,source_subdivision VARCHAR(100)
   ,source_joinkey     VARCHAR(40)  NOT NULL
   ,start_date         DATE
   ,end_date           DATE
   ,featuredetailurl   VARCHAR(255)
   ,globalid           VARCHAR(40)  NOT NULL
   ,shape              GEOMETRY
   ,CONSTRAINT wqp_src_p_pk PRIMARY KEY (permid_joinkey)
)
    """);

    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_i01 ON cipdev_owld.wqp_src_p(source_joinkey)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_i02 ON cipdev_owld.wqp_src_p(source_originator,source_featureid,source_featureid2,source_series,start_date)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_i03 ON cipdev_owld.wqp_src_p(source_originator)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_i04 ON cipdev_owld.wqp_src_p(source_featureid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_i05 ON cipdev_owld.wqp_src_p(source_featureid2)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_spx ON cipdev_owld.wqp_src_p USING gist(shape)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_src_p_u01 ON cipdev_owld.wqp_src_p(objectid)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_src_p_u02 ON cipdev_owld.wqp_src_p(globalid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_src_p_f01 ON cipdev_owld.wqp_src_p(CAST(permid_joinkey AS UUID))");

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_src_p(
    objectid
   ,permid_joinkey
   ,source_originator
   ,source_featureid
   ,source_featureid2
   ,source_joinkey
   ,start_date
   ,featuredetailurl
   ,globalid
   ,shape
)
SELECT
 a.objectid
,'{' || uuid_generate_v1() || '}'
,a.organizationidentifier
,a.monitoringlocationidentifier
,a.providername
,b.source_joinkey
,TO_DATE('""" + extract_date + """','YYYYMMDD')
,'https://www.waterqualitydata.us/provider/' || a.providername
    || '/' || a.organizationidentifier
    || '/' || a.monitoringlocationidentifier
    || '/'
,'{' || uuid_generate_v1() || '}'
,a.shape
FROM
cipdev_owld.wqp_stations a
JOIN
cipdev_owld.wqp_sfid b
ON
    a.organizationidentifier       = b.source_originator
AND a.monitoringlocationidentifier = b.source_featureid
AND a.providername                 = b.source_featureid2
WHERE
a.shape IS NOT NULL
    """);

    conn.commit();
    cursor.execute("ANALYZE cipdev_owld.wqp_src_p");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_src_p");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_src_p.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_attr");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_attr(
    objectid                            INTEGER NOT NULL
   ,source_joinkey                      VARCHAR(40) NOT NULL
   ,organizationidentifier              VARCHAR(256) NOT NULL
   ,organizationformalname              VARCHAR(256)
   ,monitoringlocationidentifier        VARCHAR(256) NOT NULL
   ,monitoringlocationname              VARCHAR(768)
   ,monitoringlocationtypename          VARCHAR(256)
   ,monitoringlocationdescription       VARCHAR(4000)
   ,huceightdigitcode                   VARCHAR(8)
   ,drainageareameasure_measureval      NUMERIC
   ,drainageareameasure_measureunt      VARCHAR(16)
   ,contributingdrainageareameasva      NUMERIC
   ,contributingdrainageareameasun      VARCHAR(16)
   ,latitudemeasure                     NUMERIC
   ,longitudemeasure                    NUMERIC
   ,sourcemapscalenumeric               VARCHAR(64)
   ,horizontalaccuracymeasureval        VARCHAR(64)
   ,horizontalaccuracymeasureunit       VARCHAR(16)
   ,horizontalcollectionmethodname      VARCHAR(2000)
   ,horizontalcoordinatereferences      VARCHAR(16)
   ,verticalmeasure_measurevalue        VARCHAR(64)
   ,verticalmeasure_measureunit         VARCHAR(16)
   ,verticalaccuracymeasurevalue        VARCHAR(64)
   ,verticalaccuracymeasureunit         VARCHAR(16)
   ,verticalcollectionmethodname        VARCHAR(2000)
   ,verticalcoordinatereferencesys      VARCHAR(16) 
   ,countrycode                         VARCHAR(2) 
   ,statecode                           VARCHAR(2) 
   ,countycode                          VARCHAR(3)
   ,aquifername                         VARCHAR(2000)
   ,localaqfrname                       VARCHAR(2000) 
   ,formationtypetext                   VARCHAR(2000)
   ,aquifertypename                     VARCHAR(2000) 
   ,constructiondatetext                VARCHAR(16) 
   ,welldepthmeasure_measurevalue       NUMERIC
   ,welldepthmeasure_measureunit        VARCHAR(16)
   ,wellholedepthmeasure_measureva      NUMERIC
   ,wellholedepthmeasure_measureun      VARCHAR(16)
   ,providername                        VARCHAR(16) NOT NULL
   ,globalid                            VARCHAR(40) NOT NULL
   ,CONSTRAINT wqp_attr_pk PRIMARY KEY(source_joinkey)
)
    """);

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_attr(
    objectid                      
   ,source_joinkey                
   ,organizationidentifier        
   ,organizationformalname        
   ,monitoringlocationidentifier  
   ,monitoringlocationname        
   ,monitoringlocationtypename    
   ,monitoringlocationdescription 
   ,huceightdigitcode             
   ,drainageareameasure_measureval
   ,drainageareameasure_measureunt
   ,contributingdrainageareameasva
   ,contributingdrainageareameasun
   ,latitudemeasure               
   ,longitudemeasure              
   ,sourcemapscalenumeric         
   ,horizontalaccuracymeasureval  
   ,horizontalaccuracymeasureunit 
   ,horizontalcollectionmethodname
   ,horizontalcoordinatereferences
   ,verticalmeasure_measurevalue  
   ,verticalmeasure_measureunit   
   ,verticalaccuracymeasurevalue  
   ,verticalaccuracymeasureunit   
   ,verticalcollectionmethodname  
   ,verticalcoordinatereferencesys
   ,countrycode                   
   ,statecode                     
   ,countycode                    
   ,aquifername                   
   ,formationtypetext             
   ,aquifertypename    
   ,localaqfrname
   ,constructiondatetext          
   ,welldepthmeasure_measurevalue 
   ,welldepthmeasure_measureunit  
   ,wellholedepthmeasure_measureva
   ,wellholedepthmeasure_measureun
   ,providername                  
   ,globalid                      
)
SELECT
 a.objectid                      
,b.source_joinkey                
,a.organizationidentifier
,a.organizationformalname
,a.monitoringlocationidentifier
,a.monitoringlocationname
,a.monitoringlocationtypename
,a.monitoringlocationdescriptiontext
,a.huceightdigitcode
,a.drainageareameasure_measurevalue
,a.drainageareameasure_measureunitcode
,a.contributingdrainageareameasure_measurevalue
,a.contributingdrainageareameasure_measureunitcode
,a.latitudemeasure
,a.longitudemeasure
,a.sourcemapscalenumeric
,a.horizontalaccuracymeasure_measurevalue
,a.horizontalaccuracymeasure_measureunitcode
,a.horizontalcollectionmethodname
,a.horizontalcoordinatereferencesystemdatumname
,a.verticalmeasure_measurevalue
,a.verticalmeasure_measureunitcode
,a.verticalaccuracymeasure_measurevalue
,a.verticalaccuracymeasure_measureunitcode
,a.verticalcollectionmethodname
,a.verticalcoordinatereferencesystemdatumname
,a.countrycode
,a.statecode
,a.countycode
,a.aquifername
,a.localaqfrname
,a.formationtypetext
,a.aquifertypename
,a.constructiondatetext
,a.welldepthmeasure_measurevalue
,a.welldepthmeasure_measureunitcode
,a.wellholedepthmeasure_measurevalue
,a.wellholedepthmeasure_measureunitcode
,a.providername       
,'{' || uuid_generate_v1() || '}'                     
FROM
cipdev_owld.wqp_stations a
JOIN
cipdev_owld.wqp_sfid b
ON
    a.organizationidentifier       = b.source_originator
AND a.monitoringlocationidentifier = b.source_featureid
AND a.providername                 = b.source_featureid2
    """);

    conn.commit();
    cursor.execute("ANALYZE cipdev_owld.wqp_attr");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_attr");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_attr.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_cip");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_cip(
    objectid                    INTEGER      NOT NULL
   ,cip_joinkey                 VARCHAR(40)  NOT NULL
   ,source_originator           VARCHAR(130) NOT NULL
   ,source_featureid            VARCHAR(100) NOT NULL
   ,source_featureid2           VARCHAR(100)
   ,source_series               VARCHAR(100)
   ,source_subdivision          VARCHAR(100)
   ,source_joinkey              VARCHAR(40)  NOT NULL
   ,permid_joinkey              VARCHAR(40)  NOT NULL
   ,start_date                  DATE
   ,end_date                    DATE
   ,cat_joinkey                 VARCHAR(40)  NOT NULL
   ,catchmentstatecode          VARCHAR(2)   NOT NULL
   ,nhdplusid                   BIGINT       NOT NULL
   ,istribal                    VARCHAR(1)   NOT NULL
   ,istribal_areasqkm           NUMERIC
   ,catchmentresolution         VARCHAR(2)   NOT NULL
   ,catchmentareasqkm           NUMERIC      NOT NULL
   ,xwalk_huc12                 VARCHAR(12)
   ,xwalk_method                VARCHAR(18)
   ,xwalk_huc12_version         VARCHAR(16)
   ,isnavigable                 VARCHAR(1)   NOT NULL
   ,hasvaa                      VARCHAR(1)   NOT NULL
   ,issink                      VARCHAR(1)   NOT NULL
   ,isheadwater                 VARCHAR(1)   NOT NULL
   ,iscoastal                   VARCHAR(1)   NOT NULL
   ,isocean                     VARCHAR(1)   NOT NULL
   ,isalaskan                   VARCHAR(1)   NOT NULL
   ,h3hexagonaddr               VARCHAR(64)
   ,globalid                    VARCHAR(40)  NOT NULL
   ,CONSTRAINT wqp_cip_pk PRIMARY KEY(cip_joinkey)
)
    """);
    
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i01 ON cipdev_owld.wqp_cip(source_joinkey)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i02 ON cipdev_owld.wqp_cip(permid_joinkey)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i03 ON cipdev_owld.wqp_cip(cat_joinkey)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i04 ON cipdev_owld.wqp_cip(xwalk_huc12)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i05 ON cipdev_owld.wqp_cip(source_originator)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i06 ON cipdev_owld.wqp_cip(source_featureid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i07 ON cipdev_owld.wqp_cip(source_featureid2)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i08 ON cipdev_owld.wqp_cip(nhdplusid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i09 ON cipdev_owld.wqp_cip(catchmentstatecode)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_i10 ON cipdev_owld.wqp_cip(catchmentresolution)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_cip_u01 ON cipdev_owld.wqp_cip(objectid)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_cip_u02 ON cipdev_owld.wqp_cip(globalid)");
    
    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_cip_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_cip_seq START WITH 1");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_src2cip");
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_src2cip( 
    objectid              INTEGER      NOT NULL
   ,src2cip_joinkey       VARCHAR(40)  NOT NULL 
   ,cip_joinkey           VARCHAR(40)  NOT NULL              
   ,source_joinkey        VARCHAR(40)  NOT NULL 
   ,permid_joinkey        VARCHAR(40)  NOT NULL 
   ,cat_joinkey           VARCHAR(40)  NOT NULL 
   ,catchmentstatecode    VARCHAR(2)   NOT NULL 
   ,nhdplusid             BIGINT       NOT NULL 
   ,catchmentresolution   VARCHAR(2)   NOT NULL
   ,cip_action            VARCHAR(255) NOT NULL
   ,overlap_measure       NUMERIC 
   ,cip_method            VARCHAR(255) 
   ,cip_parms             VARCHAR(255) 
   ,cip_date              DATE 
   ,cip_version           VARCHAR(255) 
   ,globalid              VARCHAR(40)  NOT NULL 
   ,CONSTRAINT wqp_src2cip_pk PRIMARY KEY(src2cip_joinkey)
)
    """);

    cursor.execute("CREATE UNIQUE INDEX wqp_src2cip_u01 ON cipdev_owld.wqp_src2cip(objectid)");
    cursor.execute("CREATE UNIQUE INDEX wqp_src2cip_u02 ON cipdev_owld.wqp_src2cip(globalid)");
    cursor.execute("CREATE INDEX wqp_src2cip_i01 ON cipdev_owld.wqp_src2cip(cip_joinkey)");
    cursor.execute("CREATE INDEX wqp_src2cip_i02 ON cipdev_owld.wqp_src2cip(source_joinkey)");
    cursor.execute("CREATE INDEX wqp_src2cip_i03 ON cipdev_owld.wqp_src2cip(permid_joinkey)");
    cursor.execute("CREATE INDEX wqp_src2cip_i04 ON cipdev_owld.wqp_src2cip(cat_joinkey)");
    cursor.execute("CREATE INDEX wqp_src2cip_i05 ON cipdev_owld.wqp_src2cip(catchmentstatecode)");
    cursor.execute("CREATE INDEX wqp_src2cip_i06 ON cipdev_owld.wqp_src2cip(nhdplusid)");
    cursor.execute("CREATE INDEX wqp_src2cip_i07 ON cipdev_owld.wqp_src2cip(catchmentresolution)");
    cursor.execute("CREATE INDEX wqp_src2cip_i08 ON cipdev_owld.wqp_src2cip(cip_action)");
    cursor.execute("CREATE INDEX wqp_src2cip_i09 ON cipdev_owld.wqp_src2cip(cip_date)");

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_src2cip_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_src2cip_seq START WITH 1");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
DO $$DECLARE
   rec                    RECORD;
   rec2                   RECORD;
   outint                 INTEGER;
   str_catchmentstatecode VARCHAR;
   str_cip_joinkey        VARCHAR;
   str_catjoinkey         VARCHAR;
 
BEGIN

   outint := cipsrv_engine.create_cip_batch_temp_tables();

   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,a.shape
   FROM
   cipdev_owld.wqp_src_p a
   LOOP   
      rec2 := cipsrv_nhdplus_m.index_point_simple(
          p_geometry              := rec.shape
         ,p_known_region          := NULL
         ,p_permid_joinkey        := rec.permid_joinkey::UUID
         ,p_permid_geometry       := NULL
         ,p_statesplit            := 1
      );
   
   END LOOP;
   
   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,a.shape
   ,b.nhdplusid
   ,b.catchmentstatecodes
   FROM
   cipdev_owld.wqp_src_p a
   JOIN
   tmp_cip b
   ON
   CAST(a.permid_joinkey AS UUID) = b.permid_joinkey
   LOOP
      str_cip_joinkey := '{' || uuid_generate_v1() || '}';
      str_catchmentstatecode := rec.catchmentstatecodes[1];
      str_catjoinkey := str_catchmentstatecode || rec.nhdplusid::BIGINT::VARCHAR;
      
      INSERT INTO cipdev_owld.wqp_cip(
          objectid
         ,cip_joinkey
         ,source_originator
         ,source_featureid
         ,source_featureid2
         ,source_joinkey
         ,permid_joinkey
         ,start_date
         ,cat_joinkey
         ,catchmentstatecode
         ,nhdplusid
         ,istribal
         ,istribal_areasqkm
         ,catchmentresolution
         ,catchmentareasqkm
         ,xwalk_huc12
         ,xwalk_method
         ,xwalk_huc12_version
         ,isnavigable
         ,hasvaa
         ,issink
         ,isheadwater
         ,iscoastal
         ,isocean
         ,isalaskan
         ,h3hexagonaddr
         ,globalid
      )
      SELECT
       NEXTVAL('cipdev_owld.wqp_cip_seq')
      ,str_cip_joinkey
      ,rec.source_originator
      ,rec.source_featureid
      ,rec.source_featureid2
      ,rec.source_joinkey
      ,rec.permid_joinkey
      ,rec.start_date
      ,str_catjoinkey
      ,str_catchmentstatecode
      ,rec.nhdplusid
      ,a.istribal
      ,a.istribal_areasqkm
      ,'MR'
      ,a.areasqkm AS catchmentareasqkm
      ,b.xwalk_huc12
      ,NULL
      ,'NP21'
      ,a.isnavigable
      ,a.hasvaa
      ,a.issink
      ,a.isheadwater
      ,a.iscoastal
      ,a.isocean
      ,a.isalaskan
      ,a.h3hexagonaddr
      ,'{' || uuid_generate_v1() || '}'
      FROM
      cipsrv_epageofab_m.catchment_fabric a
      LEFT JOIN (
         SELECT
          bb.nhdplusid
         ,bb.xwalk_huc12
         FROM
         cipsrv_epageofab_m.catchment_fabric_xwalk bb
         WHERE
         bb.xwalk_huc12_version = 'NP21'
      ) b
      ON
      a.nhdplusid = b.nhdplusid
      WHERE
      a.catchmentstatecode = str_catchmentstatecode
      AND a.nhdplusid = rec.nhdplusid;

      INSERT INTO cipdev_owld.wqp_src2cip( 
          objectid
         ,src2cip_joinkey
         ,cip_joinkey
         ,source_joinkey
         ,permid_joinkey 
         ,cat_joinkey 
         ,catchmentstatecode  
         ,nhdplusid   
         ,catchmentresolution
         ,cip_action
         ,overlap_measure 
         ,cip_method 
         ,cip_parms 
         ,cip_date  
         ,cip_version  
         ,globalid  
      ) VALUES (
          NEXTVAL('cipdev_owld.wqp_src2cip_seq')
         ,'{' || uuid_generate_v1() || '}'
         ,str_cip_joinkey
         ,rec.source_joinkey
         ,rec.permid_joinkey
         ,str_catjoinkey
         ,str_catchmentstatecode
         ,rec.nhdplusid
         ,'MR'
         ,'add'
         ,NULL
         ,'point_simple'
         ,NULL
         ,TO_DATE('""" + extract_date + """','YYYYMMDD')
         ,cipsrv_engine.cipsrv_version()
         ,'{' || uuid_generate_v1() || '}'
      );

   END LOOP;
   
END$$;
    """);

    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
DO $$DECLARE
   rec                    RECORD;
   rec2                   RECORD;
   outint                 INTEGER;
   str_catchmentstatecode VARCHAR;
   str_cip_joinkey        VARCHAR;
   str_catjoinkey         VARCHAR;
 
BEGIN

   outint := cipsrv_engine.create_cip_batch_temp_tables();

   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,a.shape
   FROM
   cipdev_owld.wqp_src_p a
   LOOP   
      rec2 := cipsrv_nhdplus_h.index_point_simple(
          p_geometry              := rec.shape
         ,p_known_region          := NULL
         ,p_permid_joinkey        := rec.permid_joinkey::UUID
         ,p_permid_geometry       := NULL
         ,p_statesplit            := 1
      );
   
   END LOOP;
   
   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,a.shape
   ,b.nhdplusid
   ,b.catchmentstatecodes
   FROM
   cipdev_owld.wqp_src_p a
   JOIN
   tmp_cip b
   ON
   CAST(a.permid_joinkey AS UUID) = b.permid_joinkey
   LOOP
      str_cip_joinkey := '{' || uuid_generate_v1() || '}';
      str_catchmentstatecode := rec.catchmentstatecodes[1];
      str_catjoinkey := str_catchmentstatecode || rec.nhdplusid::BIGINT::VARCHAR;
      
      INSERT INTO cipdev_owld.wqp_cip(
          objectid
         ,cip_joinkey
         ,source_originator
         ,source_featureid
         ,source_featureid2
         ,source_joinkey
         ,permid_joinkey
         ,start_date
         ,cat_joinkey
         ,catchmentstatecode
         ,nhdplusid
         ,istribal
         ,istribal_areasqkm
         ,catchmentresolution
         ,catchmentareasqkm
         ,xwalk_huc12
         ,xwalk_method
         ,xwalk_huc12_version
         ,isnavigable
         ,hasvaa
         ,issink
         ,isheadwater
         ,iscoastal
         ,isocean
         ,isalaskan
         ,h3hexagonaddr
         ,globalid
      )
      SELECT
       NEXTVAL('cipdev_owld.wqp_cip_seq')
      ,str_cip_joinkey
      ,rec.source_originator
      ,rec.source_featureid
      ,rec.source_featureid2
      ,rec.source_joinkey
      ,rec.permid_joinkey
      ,rec.start_date
      ,str_catjoinkey
      ,str_catchmentstatecode
      ,rec.nhdplusid
      ,a.istribal
      ,a.istribal_areasqkm
      ,'HR'
      ,a.areasqkm AS catchmentareasqkm
      ,b.xwalk_huc12
      ,NULL
      ,'NP21'
      ,a.isnavigable
      ,a.hasvaa
      ,a.issink
      ,a.isheadwater
      ,a.iscoastal
      ,a.isocean
      ,a.isalaskan
      ,a.h3hexagonaddr
      ,'{' || uuid_generate_v1() || '}'
      FROM
      cipsrv_epageofab_h.catchment_fabric a
      LEFT JOIN (
         SELECT
          bb.nhdplusid
         ,bb.xwalk_huc12
         FROM
         cipsrv_epageofab_h.catchment_fabric_xwalk bb
         WHERE
         bb.xwalk_huc12_version = 'NP21'
      ) b
      ON
      a.nhdplusid = b.nhdplusid
      WHERE
      a.catchmentstatecode = str_catchmentstatecode
      AND a.nhdplusid = rec.nhdplusid;

      INSERT INTO cipdev_owld.wqp_src2cip( 
          objectid
         ,src2cip_joinkey
         ,cip_joinkey
         ,source_joinkey
         ,permid_joinkey 
         ,cat_joinkey 
         ,catchmentstatecode  
         ,nhdplusid   
         ,catchmentresolution
         ,cip_action
         ,overlap_measure 
         ,cip_method 
         ,cip_parms 
         ,cip_date  
         ,cip_version  
         ,globalid  
      ) VALUES (
          NEXTVAL('cipdev_owld.wqp_src2cip_seq')
         ,'{' || uuid_generate_v1() || '}'
         ,str_cip_joinkey
         ,rec.source_joinkey
         ,rec.permid_joinkey
         ,str_catjoinkey
         ,str_catchmentstatecode
         ,rec.nhdplusid
         ,'HR'
         ,'add'
         ,NULL
         ,'point_simple'
         ,NULL
         ,TO_DATE('""" + extract_date + """','YYYYMMDD')
         ,cipsrv_engine.cipsrv_version()
         ,'{' || uuid_generate_v1() || '}'
      );

   END LOOP;
   
END$$;
    """);

    conn.commit();

    cursor.execute("ANALYZE cipdev_owld.wqp_cip");
    cursor.execute("ANALYZE cipdev_owld.wqp_src2cip");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_cip_geo");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_cip_geo(
    objectid               INTEGER     NOT NULL
   ,cat_joinkey            VARCHAR(40) NOT NULL
   ,catchmentstatecode     VARCHAR(2)  NOT NULL
   ,nhdplusid              BIGINT      NOT NULL
   ,catchmentresolution    VARCHAR     NOT NULL
   ,catchmentareasqkm      NUMERIC     NOT NULL
   ,xwalk_huc12            VARCHAR(12)
   ,xwalk_method           VARCHAR
   ,xwalk_huc12_version    VARCHAR
   ,globalid               VARCHAR(40) NOT NULL
   ,shape                  GEOMETRY
   ,CONSTRAINT wqp_cip_geo_pk PRIMARY KEY (cat_joinkey)
)
    """);

    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_geo_i01 ON cipdev_owld.wqp_cip_geo(nhdplusid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_geo_i02 ON cipdev_owld.wqp_cip_geo(catchmentstatecode)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_cip_geo_pk2 ON cipdev_owld.wqp_cip_geo(catchmentstatecode,nhdplusid)");
    cursor.execute("CREATE INDEX IF NOT EXISTS wqp_cip_geo_spx ON cipdev_owld.wqp_cip_geo USING gist(shape)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_cip_geo_u01 ON cipdev_owld.wqp_cip_geo(objectid)");
    cursor.execute("CREATE UNIQUE INDEX IF NOT EXISTS wqp_cip_geo_u02 ON cipdev_owld.wqp_cip_geo(globalid)");

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_cip_geo_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_cip_geo_seq START WITH 1");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("""
INSERT INTO cipdev_owld.wqp_cip_geo(
    objectid
   ,cat_joinkey
   ,catchmentstatecode
   ,nhdplusid
   ,catchmentresolution
   ,catchmentareasqkm
   ,xwalk_huc12
   ,xwalk_method
   ,xwalk_huc12_version
   ,globalid
   ,shape
)
SELECT
 NEXTVAL('cipdev_owld.wqp_cip_geo_seq')
,a.cat_joinkey
,a.catchmentstatecode
,a.nhdplusid
,a.catchmentresolution
,a.catchmentareasqkm
,a.xwalk_huc12
,a.xwalk_method
,a.xwalk_huc12_version
,'{' || uuid_generate_v1() || '}'
,b.shape
FROM (
   SELECT
    aa.cat_joinkey
   ,aa.catchmentstatecode
   ,aa.nhdplusid
   ,aa.catchmentresolution
   ,aa.catchmentareasqkm
   ,aa.xwalk_huc12
   ,aa.xwalk_method
   ,aa.xwalk_huc12_version
   FROM
   cipdev_owld.wqp_cip aa
   GROUP BY
    aa.cat_joinkey
   ,aa.catchmentstatecode
   ,aa.nhdplusid
   ,aa.catchmentresolution
   ,aa.catchmentareasqkm
   ,aa.xwalk_huc12
   ,aa.xwalk_method
   ,aa.xwalk_huc12_version
) a
JOIN
cipsrv_epageofab_m.catchment_fabric b
ON
    a.catchmentstatecode = b.catchmentstatecode
AND a.nhdplusid          = b.nhdplusid
WHERE
a.catchmentresolution = 'MR'
    """);
    
    conn.commit();

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_cip_geo(
    objectid
   ,cat_joinkey
   ,catchmentstatecode
   ,nhdplusid
   ,catchmentresolution
   ,catchmentareasqkm
   ,xwalk_huc12
   ,xwalk_method
   ,xwalk_huc12_version
   ,globalid
   ,shape
)
SELECT
 NEXTVAL('cipdev_owld.wqp_cip_geo_seq')
,a.cat_joinkey
,a.catchmentstatecode
,a.nhdplusid
,a.catchmentresolution
,a.catchmentareasqkm
,a.xwalk_huc12
,a.xwalk_method
,a.xwalk_huc12_version
,'{' || uuid_generate_v1() || '}'
,b.shape
FROM (
   SELECT
    aa.cat_joinkey
   ,aa.catchmentstatecode
   ,aa.nhdplusid
   ,aa.catchmentresolution
   ,aa.catchmentareasqkm
   ,aa.xwalk_huc12
   ,aa.xwalk_method
   ,aa.xwalk_huc12_version
   FROM
   cipdev_owld.wqp_cip aa
   GROUP BY
    aa.cat_joinkey
   ,aa.catchmentstatecode
   ,aa.nhdplusid
   ,aa.catchmentresolution
   ,aa.catchmentareasqkm
   ,aa.xwalk_huc12
   ,aa.xwalk_method
   ,aa.xwalk_huc12_version
) a
JOIN
cipsrv_epageofab_h.catchment_fabric b
ON
    a.catchmentstatecode = b.catchmentstatecode
AND a.nhdplusid = b.nhdplusid
WHERE
a.catchmentresolution = 'HR'
ON CONFLICT DO NOTHING
    """);
    
    conn.commit();
    
    cursor.execute("ANALYZE cipdev_owld.wqp_cip_geo");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_cip_geo");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_cip_geo.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_huc12");
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_huc12(
    objectid                        INTEGER      NOT NULL
   ,source_originator               VARCHAR(130) NOT NULL
   ,source_featureid                VARCHAR(100) NOT NULL
   ,source_featureid2               VARCHAR(100)
   ,source_series                   VARCHAR(100)
   ,source_subdivison               VARCHAR(100)
   ,source_joinkey                  VARCHAR(40)  NOT NULL
   ,permid_joinkey                  VARCHAR(40)  NOT NULL
   ,start_date                      DATE
   ,end_date                        DATE
   ,xwalk_huc12                     VARCHAR(12)  NOT NULL
   ,xwalk_catresolution             VARCHAR(2)   NOT NULL
   ,xwalk_huc12_version             VARCHAR(16)  NOT NULL
   ,xwalk_huc12_areasqkm            NUMERIC
   ,globalid                        VARCHAR(40)  NOT NULL
)
    """);

    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_pk  ON cipdev_owld.wqp_huc12(permid_joinkey,xwalk_huc12,xwalk_catresolution)");
    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_u02 ON cipdev_owld.wqp_huc12(globalid)");
    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_uk  ON cipdev_owld.wqp_huc12(objectid)");
    cursor.execute("CREATE INDEX wqp_huc12_i01 ON cipdev_owld.wqp_huc12(source_joinkey)");
    cursor.execute("CREATE INDEX wqp_huc12_i02 ON cipdev_owld.wqp_huc12(permid_joinkey)");
    cursor.execute("CREATE INDEX wqp_huc12_i03 ON cipdev_owld.wqp_huc12(source_originator)");
    cursor.execute("CREATE INDEX wqp_huc12_i04 ON cipdev_owld.wqp_huc12(source_featureid)");
    cursor.execute("CREATE INDEX wqp_huc12_i05 ON cipdev_owld.wqp_huc12(source_featureid2)");
    cursor.execute("CREATE INDEX wqp_huc12_i06 ON cipdev_owld.wqp_huc12(xwalk_huc12)");
    cursor.execute("CREATE INDEX wqp_huc12_i07 ON cipdev_owld.wqp_huc12(xwalk_catresolution)");

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_huc12_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_huc12_seq START WITH 1");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("""
INSERT INTO cipdev_owld.wqp_huc12(
    objectid
   ,source_originator
   ,source_featureid
   ,source_featureid2
   ,source_joinkey
   ,permid_joinkey
   ,start_date
   ,xwalk_huc12
   ,xwalk_catresolution
   ,xwalk_huc12_version
   ,xwalk_huc12_areasqkm
   ,globalid
)
SELECT
 NEXTVAL('cipdev_owld.wqp_huc12_seq')
,a.source_originator
,a.source_featureid
,a.source_featureid2
,a.source_joinkey
,a.permid_joinkey
,a.start_date
,a.xwalk_huc12
,'MR'
,'NP21'
,b.areasqkm
,'{' || uuid_generate_v1() || '}'
FROM (
   SELECT
    aa.source_originator
   ,aa.source_featureid
   ,aa.source_featureid2
   ,aa.source_joinkey
   ,aa.permid_joinkey
   ,aa.start_date
   ,aa.xwalk_huc12
   FROM
   cipdev_owld.wqp_cip aa
   WHERE
       aa.catchmentresolution = 'MR'
   AND aa.xwalk_huc12 IS NOT NULL
   GROUP BY
    aa.source_originator
   ,aa.source_featureid
   ,aa.source_featureid2
   ,aa.source_joinkey
   ,aa.permid_joinkey
   ,aa.start_date
   ,aa.xwalk_huc12
) a
JOIN
cipsrv_epageofab_m.catchment_fabric_huc12_np21 b
ON
a.xwalk_huc12 = b.xwalk_huc12
    """);

    conn.commit();

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_huc12(
    objectid
   ,source_originator
   ,source_featureid
   ,source_featureid2
   ,source_joinkey
   ,permid_joinkey
   ,start_date
   ,xwalk_huc12
   ,xwalk_catresolution
   ,xwalk_huc12_version
   ,xwalk_huc12_areasqkm
   ,globalid
)
SELECT
 nextval('cipdev_owld.wqp_huc12_seq')
,a.source_originator
,a.source_featureid
,a.source_featureid2
,a.source_joinkey
,a.permid_joinkey
,a.start_date
,a.xwalk_huc12
,'HR'
,'NP21'
,b.areasqkm
,'{' || uuid_generate_v1() || '}'
FROM (
   SELECT
    aa.source_originator
   ,aa.source_featureid
   ,aa.source_featureid2
   ,aa.source_joinkey
   ,aa.permid_joinkey
   ,aa.start_date
   ,aa.xwalk_huc12
   FROM
   cipdev_owld.wqp_cip aa
   WHERE
       aa.catchmentresolution = 'HR'
   AND aa.xwalk_huc12 IS NOT NULL
   GROUP BY
    aa.source_originator
   ,aa.source_featureid
   ,aa.source_featureid2
   ,aa.source_joinkey
   ,aa.permid_joinkey
   ,aa.start_date
   ,aa.xwalk_huc12
) a
JOIN
cipsrv_epageofab_h.catchment_fabric_huc12_np21 b
ON
a.xwalk_huc12 = b.xwalk_huc12
    """);

    conn.commit();

    cursor.execute("ANALYZE cipdev_owld.wqp_huc12");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_huc12");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_huc12.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_huc12_geo");
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_huc12_geo(
    objectid                        INTEGER      NOT NULL
   ,xwalk_huc12                     VARCHAR(12)  NOT NULL
   ,xwalk_catresolution             VARCHAR(2)   NOT NULL
   ,xwalk_huc12_version             VARCHAR(16)  NOT NULL
   ,xwalk_huc12_areasqkm            NUMERIC      NOT NULL
   ,globalid                        VARCHAR(40)  NOT NULL
   ,shape                           GEOMETRY
)
    """);

    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_geo_pk  ON cipdev_owld.wqp_huc12_geo(xwalk_catresolution,xwalk_huc12)");
    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_geo_u02 ON cipdev_owld.wqp_huc12_geo(globalid)");
    cursor.execute("CREATE UNIQUE INDEX wqp_huc12_geo_uk  ON cipdev_owld.wqp_huc12_geo(objectid)");
    cursor.execute("CREATE INDEX wqp_huc12_geo_01i        ON cipdev_owld.wqp_huc12_geo(xwalk_catresolution)");
    cursor.execute("CREATE INDEX wqp_huc12_geo_02i        ON cipdev_owld.wqp_huc12_geo(xwalk_huc12)");

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_huc12_geo_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_huc12_geo_seq START WITH 1");

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_huc12_geo(
    objectid
   ,xwalk_huc12
   ,xwalk_catresolution
   ,xwalk_huc12_version
   ,xwalk_huc12_areasqkm
   ,globalid
   ,shape
)
SELECT
 NEXTVAL('cipdev_owld.wqp_huc12_geo_seq')
,a.xwalk_huc12
,a.xwalk_catresolution
,a.xwalk_huc12_version
,b.areasqkm
,'{' || uuid_generate_v1() || '}'
,b.shape
FROM (
   SELECT
    aa.xwalk_huc12
   ,aa.xwalk_catresolution
   ,aa.xwalk_huc12_version
   FROM
   cipdev_owld.wqp_huc12 aa
   WHERE
       aa.xwalk_catresolution = 'MR'
   AND aa.xwalk_huc12 IS NOT NULL
   GROUP BY
    aa.xwalk_huc12
   ,aa.xwalk_catresolution
   ,aa.xwalk_huc12_version
) a
JOIN
cipsrv_epageofab_m.catchment_fabric_huc12_np21 b
ON
a.xwalk_huc12 = b.xwalk_huc12
    """);

    conn.commit();

    cursor.execute("""
INSERT INTO cipdev_owld.wqp_huc12_geo(
    objectid
   ,xwalk_huc12
   ,xwalk_catresolution
   ,xwalk_huc12_version
   ,xwalk_huc12_areasqkm
   ,globalid
   ,shape
)
SELECT
 NEXTVAL('cipdev_owld.wqp_huc12_geo_seq')
,a.xwalk_huc12
,a.xwalk_catresolution
,a.xwalk_huc12_version
,b.areasqkm
,'{' || uuid_generate_v1() || '}'
,b.shape
FROM (
   SELECT
    aa.xwalk_huc12
   ,aa.xwalk_catresolution
   ,aa.xwalk_huc12_version
   FROM
   cipdev_owld.wqp_huc12 aa
   WHERE
       aa.xwalk_catresolution = 'HR'
   AND aa.xwalk_huc12 IS NOT NULL
   GROUP BY
    aa.xwalk_huc12
   ,aa.xwalk_catresolution
   ,aa.xwalk_huc12_version
) a
JOIN
cipsrv_epageofab_h.catchment_fabric_huc12_np21 b
ON
a.xwalk_huc12 = b.xwalk_huc12
    """);

    conn.commit();

    cursor.execute("ANALYZE cipdev_owld.wqp_huc12_geo");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_huc12_geo");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_huc12_geo.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_rad_p");
    cursor.execute("""
CREATE TABLE cipdev_owld.wqp_rad_p(
    objectid                        INTEGER      NOT NULL
   ,permanent_identifier            VARCHAR(40)  NOT NULL
   ,eventdate                       DATE
   ,reachcode                       VARCHAR(14)
   ,reachsmdate                     DATE
   ,reachresolution                 VARCHAR(255)
   ,feature_permanent_identifier    VARCHAR(40)
   ,featureclassref                 INTEGER
   ,source_originator               VARCHAR(130) NOT NULL
   ,source_featureid                VARCHAR(100) NOT NULL
   ,source_featureid2               VARCHAR(100)
   ,source_datadesc                 VARCHAR(100)
   ,source_series                   VARCHAR(100)
   ,source_subdivision              VARCHAR(100)
   ,source_joinkey                  VARCHAR(40)  NOT NULL
   ,permid_joinkey                  VARCHAR(40)  NOT NULL
   ,start_date                      DATE
   ,end_date                        DATE
   ,featuredetailurl                VARCHAR(255)
   ,measure                         NUMERIC
   ,eventtype                       INTEGER
   ,eventoffset                     NUMERIC
   ,geogstate                       VARCHAR(2)
   ,xwalk_huc12                     VARCHAR(12)
   ,xwalk_method                    VARCHAR(16)
   ,xwalk_huc12_version             VARCHAR(16)
   ,isnavigable                     VARCHAR(1)
   ,hasvaa                          VARCHAR(1)
   ,issink                          VARCHAR(1)
   ,isheadwater                     VARCHAR(1)
   ,iscoastal                       VARCHAR(1)
   ,isocean                         VARCHAR(1)
   ,isalaskan                       VARCHAR(1)
   ,h3hexagonaddr                   VARCHAR(1)
   ,globalid                        VARCHAR(40)  NOT NULL
   ,shape                           GEOMETRY
)
    """);

    cursor.execute("CREATE UNIQUE INDEX wqp_rad_p_pk  ON cipdev_owld.wqp_rad_p(permanent_identifier)");
    cursor.execute("CREATE UNIQUE INDEX wqp_rad_p_u02 ON cipdev_owld.wqp_rad_p(globalid)");
    cursor.execute("CREATE UNIQUE INDEX wqp_rad_p_uk  ON cipdev_owld.wqp_rad_p(objectid)");
    cursor.execute("CREATE INDEX wqp_rad_p_i01        ON cipdev_owld.wqp_rad_p(source_joinkey)");
    cursor.execute("CREATE INDEX wqp_rad_p_i03        ON cipdev_owld.wqp_rad_p(source_originator)");
    cursor.execute("CREATE INDEX wqp_rad_p_i04        ON cipdev_owld.wqp_rad_p(source_featureid)");
    cursor.execute("CREATE INDEX wqp_rad_p_i05        ON cipdev_owld.wqp_rad_p(source_featureid2)");
    cursor.execute("CREATE INDEX wqp_rad_p_i06        ON cipdev_owld.wqp_rad_p(reachresolution)");

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_rad_p_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_rad_p_seq START WITH 1");

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
DO $$DECLARE
   rec                    RECORD;
   rec2                   RECORD;
   outint                 INTEGER;
   str_catchmentstatecode VARCHAR;
   str_cip_joinkey        VARCHAR;
   str_catjoinkey         VARCHAR;
 
BEGIN

   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,b.catchmentstatecode
   ,b.nhdplusid
   ,b.xwalk_huc12
   ,b.isnavigable
   ,b.hasvaa
   ,b.issink
   ,b.isheadwater
   ,b.iscoastal
   ,b.isalaskan
   ,b.h3hexagonaddr
   ,a.shape
   FROM
   cipdev_owld.wqp_src_p a
   JOIN
   cipdev_owld.wqp_cip b
   ON
   a.permid_joinkey = b.permid_joinkey
   WHERE 
      b.catchmentresolution = 'MR'
   AND ( b.isnavigable = 'Y' OR b.iscoastal   = 'Y' )
   LOOP
      rec2 := cipsrv_nhdplus_m.catconstrained_index(
          p_point                     := rec.shape
         ,p_return_link_path          := FALSE
         ,p_known_region              := NULL
         ,p_known_catchment_nhdplusid := rec.nhdplusid
      );
      
      IF rec2.out_return_code = 0
      THEN
         INSERT INTO cipdev_owld.wqp_rad_p(
             objectid
            ,permanent_identifier
            ,eventdate
            ,reachcode
            ,reachsmdate
            ,reachresolution
            ,feature_permanent_identifier
            ,source_originator
            ,source_featureid
            ,source_featureid2
            ,source_joinkey
            ,permid_joinkey
            ,start_date
            ,featuredetailurl
            ,measure 
            ,eventtype
            ,eventoffset
            ,geogstate
            ,xwalk_huc12
            ,isnavigable
            ,hasvaa
            ,issink
            ,isheadwater
            ,iscoastal
            ,isalaskan
            ,h3hexagonaddr
            ,globalid
            ,shape
         ) VALUES (
             nextval('cipdev_owld.wqp_rad_p_seq')
            ,'{' || uuid_generate_v1() || '}'
            ,rec.start_date
            ,rec2.out_reachcode
            ,NULL
            ,'MR'
            ,rec2.out_nhdplusid::VARCHAR
            ,rec.source_originator             
            ,rec.source_featureid              
            ,rec.source_featureid2                            
            ,rec.source_joinkey
            ,rec.permid_joinkey             
            ,rec.start_date                                       
            ,rec.featuredetailurl
            ,rec2.out_snap_measure
            ,10032
            ,0
            ,rec.catchmentstatecode
            ,rec.xwalk_huc12
            ,rec.isnavigable
            ,rec.hasvaa
            ,rec.issink
            ,rec.isheadwater
            ,rec.iscoastal
            ,rec.isalaskan
            ,rec.h3hexagonaddr
            ,'{' || uuid_generate_v1() || '}'
            ,rec2.out_end_point
         );
  
      END IF;
      
   END LOOP;
   
END$$;
    """);

    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("""
DO $$DECLARE
   rec                    RECORD;
   rec2                   RECORD;
   outint                 INTEGER;
   str_catchmentstatecode VARCHAR;
   str_cip_joinkey        VARCHAR;
   str_catjoinkey         VARCHAR;
 
BEGIN

   FOR rec IN
   SELECT
    a.objectid
   ,a.permid_joinkey
   ,a.source_originator
   ,a.source_featureid
   ,a.source_featureid2
   ,a.source_joinkey
   ,a.start_date
   ,a.featuredetailurl
   ,b.catchmentstatecode
   ,b.nhdplusid
   ,b.xwalk_huc12
   ,b.isnavigable
   ,b.hasvaa
   ,b.issink
   ,b.isheadwater
   ,b.iscoastal
   ,b.isalaskan
   ,b.h3hexagonaddr
   ,a.shape
   FROM
   cipdev_owld.wqp_src_p a
   JOIN
   cipdev_owld.wqp_cip b
   ON
   a.permid_joinkey = b.permid_joinkey
   WHERE 
      b.catchmentresolution = 'HR'
   AND ( b.isnavigable = 'Y' OR b.iscoastal   = 'Y' )
   LOOP
      rec2 := cipsrv_nhdplus_h.catconstrained_index(
          p_point                     := rec.shape
         ,p_return_link_path          := FALSE
         ,p_known_region              := NULL
         ,p_known_catchment_nhdplusid := rec.nhdplusid
      );
      
      IF rec2.out_return_code = 0
      THEN
         INSERT INTO cipdev_owld.wqp_rad_p(
             objectid
            ,permanent_identifier
            ,eventdate
            ,reachcode
            ,reachsmdate
            ,reachresolution
            ,feature_permanent_identifier
            ,source_originator
            ,source_featureid
            ,source_featureid2
            ,source_joinkey
            ,permid_joinkey
            ,start_date
            ,featuredetailurl
            ,measure 
            ,eventtype
            ,eventoffset
            ,geogstate
            ,xwalk_huc12
            ,isnavigable
            ,hasvaa
            ,issink
            ,isheadwater
            ,iscoastal
            ,isalaskan
            ,h3hexagonaddr
            ,globalid
            ,shape
         ) VALUES (
             nextval('cipdev_owld.wqp_rad_p_seq')
            ,'{' || uuid_generate_v1() || '}'
            ,rec.start_date
            ,rec2.out_reachcode
            ,NULL
            ,'HR'
            ,rec2.out_nhdplusid::VARCHAR
            ,rec.source_originator             
            ,rec.source_featureid              
            ,rec.source_featureid2                            
            ,rec.source_joinkey
            ,rec.permid_joinkey             
            ,rec.start_date                                       
            ,rec.featuredetailurl
            ,rec2.out_snap_measure
            ,10032
            ,0
            ,rec.catchmentstatecode
            ,rec.xwalk_huc12
            ,rec.isnavigable
            ,rec.hasvaa
            ,rec.issink
            ,rec.isheadwater
            ,rec.iscoastal
            ,rec.isalaskan
            ,rec.h3hexagonaddr
            ,'{' || uuid_generate_v1() || '}'
            ,rec2.out_end_point
         );
  
      END IF;
      
   END LOOP;
   
END$$
    """);

    conn.commit();

    cursor.execute("ANALYZE cipdev_owld.wqp_rad_p");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_rad_p.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

cnt = 0;
with closing(conn.cursor()) as cursor:
    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_rad_metadata");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_rad_metadata(
    objectid                      INTEGER     NOT NULL
   ,meta_processid                VARCHAR(40) NOT NULL
   ,processdescription            VARCHAR(4000)
   ,processdate                   TIMESTAMP WITH TIME ZONE
   ,attributeaccuracyreport       VARCHAR(1800)
   ,logicalconsistencyreport      VARCHAR(1000)
   ,completenessreport            VARCHAR(2400)
   ,horizpositionalaccuracyreport VARCHAR(3100)
   ,vertpositionalaccuracyreport  VARCHAR(3100)
   ,metadatastandardname          VARCHAR(100)
   ,metadatastandardversion       VARCHAR(40)
   ,metadatadate                  TIMESTAMP WITH TIME ZONE
   ,datasetcredit                 VARCHAR(4000)
   ,contactorganization           VARCHAR(100)
   ,addresstype                   VARCHAR(40)
   ,address                       VARCHAR(100)
   ,city                          VARCHAR(40)
   ,stateorprovince               VARCHAR(30)
   ,postalcode                    VARCHAR(20)
   ,contactvoicetelephone         VARCHAR(40)
   ,contactinstructions           VARCHAR(120)
   ,contactemailaddress           VARCHAR(40)
   ,globalid                      VARCHAR(40)   NOT NULL
   ,CONSTRAINT wqp_rad_metadata_pk PRIMARY KEY (meta_processid)
   ,CONSTRAINT wqp_rad_metadata_u01 UNIQUE (objectid)
   ,CONSTRAINT wqp_rad_metadata_u02 UNIQUE (globalid)
)
    """);

    cursor.execute("DROP TABLE IF EXISTS cipdev_owld.wqp_rad_evt2meta");
    cursor.execute("""
CREATE TABLE IF NOT EXISTS cipdev_owld.wqp_rad_evt2meta(
    objectid                      INTEGER
   ,permanent_identifier          VARCHAR(40)  NOT NULL
   ,meta_processid                VARCHAR(40)  NOT NULL
   ,reachresolution               VARCHAR(2) 
   ,globalid                      VARCHAR(40)  NOT NULL
   ,CONSTRAINT wqp_rad_evt2meta_pk PRIMARY KEY (meta_processid, permanent_identifier)
   ,CONSTRAINT wqp_rad_evt2meta_u01 UNIQUE (objectid)
   ,CONSTRAINT wqp_rad_evt2meta_u02 UNIQUE (globalid)
)
    """);

    cursor.execute("DROP SEQUENCE IF EXISTS cipdev_owld.wqp_rad_evt2meta_seq");
    cursor.execute("CREATE SEQUENCE cipdev_owld.wqp_rad_evt2meta_seq START WITH 1");

    cursor.execute("""
DO $$DECLARE
   str_metadata_processid VARCHAR(40);
 
BEGIN

   str_metadata_processid := '{' || uuid_generate_v1() || '}';
   
   INSERT INTO cipdev_owld.wqp_rad_metadata(
       objectid
      ,meta_processid
      ,processdate
      ,contactorganization
      ,address
      ,city
      ,stateorprovince
      ,postalcode 
      ,contactinstructions
      ,globalid 
   ) VALUES (
       1
      ,str_metadata_processid
      ,TO_DATE('""" + extract_date + """','YYYYMMDD')
      ,'US EPA Office of Water'
      ,'1200 Pennsylvania Avenue, N.W.'
      ,'Washington'
      ,'DC'
      ,'20460'
      ,'visit https://www.epa.gov/aboutepa/forms/contact-epas-office-water'
      ,'{' || uuid_generate_v1() || '}'
   );
   
   INSERT INTO cipdev_owld.wqp_rad_evt2meta(
	    objectid
      ,permanent_identifier
      ,meta_processid
      ,reachresolution
      ,globalid
   )
	SELECT
    NEXTVAL('cipdev_owld.wqp_rad_evt2meta_seq')::INTEGER
   ,a.permanent_identifier
   ,str_metadata_processid
   ,a.reachresolution
   ,'{' || uuid_generate_v1() || '}'
   FROM
   cipdev_owld.wqp_rad_p a;

END$$
    """);

    conn.commit();

    cursor.execute("ANALYZE cipdev_owld.wqp_rad_evt2meta");

    cursor.execute("SELECT COUNT(*) FROM cipdev_owld.wqp_rad_evt2meta");
    cnt = cursor.fetchone()[0];

print("loaded " + str(cnt) + " records into cipdev_owld.wqp_rad_evt2meta.");
print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
UPDATE cipdev_owld.wqp_sfid a
SET
 cat_mr_count           = (SELECT COUNT(*) FROM cipdev_owld.wqp_cip   aa WHERE aa.source_joinkey = a.source_joinkey AND aa.catchmentresolution = 'MR') 
,cat_hr_count           = (SELECT COUNT(*) FROM cipdev_owld.wqp_cip   bb WHERE bb.source_joinkey = a.source_joinkey AND bb.catchmentresolution = 'HR')
,xwalk_huc12_np21_count = (SELECT COUNT(*) FROM cipdev_owld.wqp_huc12 cc WHERE cc.source_joinkey = a.source_joinkey AND cc.xwalk_huc12_version = 'NP21')
,xwalk_huc12_nphr_count = 0
,xwalk_huc12_f3_count   = 0
,rad_mr_event_count     = (SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p dd WHERE dd.source_joinkey = a.source_joinkey AND dd.reachresolution = 'MR')
,rad_hr_event_count     = (SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p ee WHERE ee.source_joinkey = a.source_joinkey AND ee.reachresolution = 'HR')
,rad_mr_point_count     = (SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p ff WHERE ff.source_joinkey = a.source_joinkey AND ff.reachresolution = 'MR')
,rad_hr_point_count     = (SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p gg WHERE gg.source_joinkey = a.source_joinkey AND gg.reachresolution = 'HR')
,rad_mr_line_count      = 0
,rad_hr_line_count      = 0
,rad_mr_area_count      = 0
,rad_hr_area_count      = 0
    """);

    conn.commit();

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");

In [ ]:
%%time

with closing(conn.cursor()) as cursor:
    cursor.execute("""
        SELECT
         'wqp_attr' AS tbl
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_attr) AS orig
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_attr) AS repl
        UNION ALL SELECT
         'wqp_cip'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_cip)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_cip)
        UNION ALL SELECT
         'wqp_cip_geo'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_cip_geo)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_cip_geo)
        UNION ALL SELECT
         'wqp_control'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_control)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_control)
        UNION ALL SELECT
         'wqp_huc12'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_huc12)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_huc12)
        UNION ALL SELECT
         'wqp_huc12_geo'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_huc12_geo)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_huc12_geo)
        UNION ALL SELECT
         'wqp_rad_evt2meta'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_rad_evt2meta)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_rad_evt2meta)
        UNION ALL SELECT
         'wqp_rad_p'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_rad_p)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_rad_p)
        UNION ALL SELECT
         'wqp_sfid'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_sfid)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_sfid)
        UNION ALL SELECT
         'wqp_srccip'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_src2cip)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_src2cip)
        UNION ALL SELECT
         'wqp_src_p'
        ,(SELECT COUNT(*) FROM cipsrv_owld.wqp_src_p)
        ,(SELECT COUNT(*) FROM cipdev_owld.wqp_src_p)
        
    """);

    for row in cursor:
        print(f"{row[0]} orig:{row[1]} repl:{row[2]}")

print(f"{datetime.datetime.now().strftime('%m/%d/%Y %H:%M:%S')}");